# Guanajuato SourceBuilder — reproducibility and QA

Reference coordinate: **20.4243195, -101.1466639** (not a verified parcel). This notebook does not modify existing run outputs; commands to fetch NASA/ISRIC data are opt-in and deliberately disabled by default.

Run from the repository root in the Python 3.11 environment with `source` dependencies installed.

In [ ]:
from pathlib import Path
import json
from gradientdescentfarmers.source_builder.adapter import REPO_ROOT, NASAProfileSourceBuilder, load_source_config
from gradientdescentfarmers.source_builder.core.audit import audit_profile

print('Repository:', REPO_ROOT)
assert (REPO_ROOT/'resources/guanajuato_reference.json').exists()
farmer = json.loads((REPO_ROOT/'resources/guanajuato_reference.json').read_text())
assert (farmer['latitude'], farmer['longitude']) == (20.4243195, -101.1466639)
print('Farm ID:', farmer['farm_id'])

## 1. Preview: schema and contract validation without remote requests

This exports an **unpopulated schema manifest**, not a validated climate dataset.

In [ ]:
cfg=load_source_config(REPO_ROOT/'resources/guanajuato_integrated.json')
cfg['run']=False
cfg['output_root']='data/previews'
preview=NASAProfileSourceBuilder().build(cfg)
print(json.dumps({k:v for k,v in preview.items() if k not in ('tables','model_readiness')},indent=2))
assert preview['source_status']['nasa_power']=='not_requested'


## 2. Integrated retrieval (explicit opt-in)

Set `RUN_INTEGRATED=True` manually only when a new acquisition is needed.
Sources: NASA POWER, SRTM, SoilGrids, MODIS (3 products), GPM, SMAP and World Bank.
Each run is stored under `data/integrated_guanajuato/guanajuato_valle_demo/runs/<run_id>`.
The published reference run is `67b66942681a4214`; it can be consumed without another acquisition.


In [ ]:
RUN_INTEGRATED=False
integrated=None
if RUN_INTEGRATED:
    cfg=load_source_config(REPO_ROOT/'resources/guanajuato_integrated.json')
    cfg['run']=True
    integrated=NASAProfileSourceBuilder().build(cfg)
    print(integrated['run_id'],integrated['build_state'])
else:
    print('Skipped real network calls')


## 3. Integrated run audit

Reads saved manifests and verifies SHA-256. This does not download data or rewrite source tables.


In [ ]:
from gradientdescentfarmers.source_builder.core.audit import audit_sites
root=REPO_ROOT/'data/integrated_guanajuato'
latest=root/farmer['farm_id']/'latest.json'
if latest.exists():
    report=audit_sites(root,sites=(farmer['farm_id'],))
    print(json.dumps(report,indent=2,default=str)[:5000])
else:
    print('No local integrated run; download the approved bundle before auditing')


## 4. Earthdata smoke test (explicit opt-in)

Requires Earthdata credentials. Two GPM half-hours are **not a full day**. Small SMAP window. Review `availability.json`, table QA and extraction warnings. Original HDF5 cache should not be deleted.

In [ ]:
RUN_SMOKE=False
if RUN_SMOKE:
    from gradientdescentfarmers.source_builder.core.settings import SOURCE_TABLES
    cfg=load_source_config(REPO_ROOT/'resources/guanajuato_integrated.json')
    settings=json.loads((REPO_ROOT/cfg.pop('settings_file')).read_text())
    settings['source_overrides']={name:name in {'gpm_imerg_v07','smap_l4'} for name in SOURCE_TABLES}
    settings.update(gpm_end='2024-07-01T00:59:59',gpm_max_granules=2)
    cfg.update(settings=settings,output_root='data/diagnostics/guanajuato_smoke',run=True)
    smoke=NASAProfileSourceBuilder().build(cfg)
    print(smoke['run_id'],smoke['source_status'].get('gpm_imerg_v07'),smoke['source_status'].get('smap_l4'))
else:
    print('Earthdata smoke skipped')


## 5. Exact GPM slots and integrated handoff (read-only)

The reference run contains the 48 exact UTC half-hours of 2024-07-01.
Inspect the persisted data and coverage using the source handoff gate.
This does not authorize a PCSE simulation or trigger another download.


In [ ]:
from gradientdescentfarmers.source_builder.core.handoff import assess_run
run_dir=REPO_ROOT/'data/integrated_guanajuato/guanajuato_valle_demo/runs/67b66942681a4214'
if (run_dir/'farm_profile.json').exists():
    handoff=assess_run(run_dir)
    print(json.dumps({key:handoff[key] for key in ('run_id','integrity_ok','weather_source_ready','modis','gpm_complete_utc_day','smap_snapshot_available','wofost_pp_input_ready','wofost_wlp_input_ready')},indent=2))
else:
    print('Download the approved reference run to inspect its source handoff')


## 6. Offline scientific regression

From terminal: `python -m pytest tests/source_builder tests/test_scaffold.py -q`. Compare QA fields and archived provenance; never claim observed yield/predicted profit from baseline weather/soil.